# Pipeline Diário de Ameaças com KEV CISA

Este notebook produz uma base diária de ameaças a partir do catálogo KEV da CISA com uma decisão central de arquitetura: manter o fluxo principal simples, determinístico e barato de operar.

## Por que este desenho

- KEV é a fonte primária porque já representa CVEs com exploração observada em ambiente real.
- `dateAdded` é o eixo temporal principal porque registra quando a ameaça entrou formalmente no catálogo operacional da CISA.
- O modo padrão é KEV-only para evitar dependência de APIs externas no fluxo diário.
- NVD e EPSS entram como enriquecimentos opcionais, isolados, para não degradar a rotina principal quando houver latência, rate limit ou indisponibilidade.

## Objetivo

Gerar CSVs prontos para monitoramento, priorização e análise diária sem depender de modelagem em grafo ou junções intermediárias frágeis.

## Modos de operação

- Padrão operacional: `PIPELINE_MODE="kev"` e `RUN_NVD=False`
- Enriquecimento técnico opcional: `PIPELINE_MODE="full"` e `RUN_NVD=True`
- Enriquecimento probabilístico opcional: `RUN_EPSS=True`

## Saídas principais

- `kev_raw.csv`: snapshot bruto do catálogo KEV
- `threats_daily_events.csv`: base canônica de eventos diários
- `threats_daily_counts.csv`: série diária agregada

## Saídas analíticas auxiliares

- `threats_by_vendor.csv`
- `threats_by_product.csv`
- `summary.json`

## Saídas opcionais de enriquecimento

- `enrich_nvd.csv`
- `enrich_epss.csv`
- `threats_daily_enriched.csv`

## Ordem recomendada de execução

1. Célula 2: configuração
2. Célula 3: funções de coleta, normalização e exportação
3. Célula 4: execução principal
4. Célula 6: análise e validação
5. Célula 7: visualizações
6. Célula 8: resumo final
7. Célula 9: leitura das decisões de enriquecimento opcional
8. Célula 10: validação da seção opcional

In [12]:
# 1) configuração do pipeline
import json
import time
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
import requests

# Modo principal: "kev" (padrão) ou "full" (habilita enriquecimento opcional)
PIPELINE_MODE = "kev"
RUN_NVD = False   # Só tem efeito quando PIPELINE_MODE == "full"
RUN_EPSS = False  # Enriquecimento opcional de probabilidade de exploração

OUT_DIR = Path(".")
PLOTS_DIR = OUT_DIR / "plots"
PLOTS_DIR.mkdir(exist_ok=True)

FILES = {
    "kev_raw": OUT_DIR / "kev_raw.csv",
    "threats_daily_events": OUT_DIR / "threats_daily_events.csv",
    "threats_daily_counts": OUT_DIR / "threats_daily_counts.csv",
    "threats_by_vendor": OUT_DIR / "threats_by_vendor.csv",
    "threats_by_product": OUT_DIR / "threats_by_product.csv",
    "enrich_nvd": OUT_DIR / "enrich_nvd.csv",
    "enrich_epss": OUT_DIR / "enrich_epss.csv",
    "threats_daily_enriched": OUT_DIR / "threats_daily_enriched.csv",
    "summary": OUT_DIR / "summary.json",
}

KEV_CSV_URL = "https://www.cisa.gov/sites/default/files/csv/known_exploited_vulnerabilities.csv"
NVD_API_URL = "https://services.nvd.nist.gov/rest/json/cves/2.0"
EPSS_API_URL = "https://api.first.org/data/v1/epss"
NVD_API_KEY = ""  # Opcional

if PIPELINE_MODE not in {"kev", "full"}:
    raise ValueError("PIPELINE_MODE inválido. Use 'kev' ou 'full'.")

if not isinstance(RUN_NVD, bool) or not isinstance(RUN_EPSS, bool):
    raise ValueError("RUN_NVD e RUN_EPSS devem ser booleanos (True/False).")

if PIPELINE_MODE == "kev" and RUN_NVD:
    print("Aviso: RUN_NVD foi desativado automaticamente (modo KEV-only).")
    RUN_NVD = False

print("Configuração concluída.")
print(f"Modo de execução: {PIPELINE_MODE}")
print(f"Enriquecimento NVD ativo: {RUN_NVD}")
print(f"Enriquecimento EPSS ativo: {RUN_EPSS}")
print(f"Diretório de saída: {OUT_DIR.resolve()}")

Configuração concluída.
Modo de execução: kev
Enriquecimento NVD ativo: False
Enriquecimento EPSS ativo: False
Diretório de saída: /Users/gabe/graphos


In [29]:
# 2) funções de coleta, normalização e exportação

import re
from urllib.parse import urlparse


OFFICIAL_ADVISORY_DOMAINS = (
    "cisa.gov",
    "microsoft.com",
    "adobe.com",
    "apple.com",
    "cisco.com",
    "oracle.com",
    "vmware.com",
    "fortinet.com",
    "paloaltonetworks.com",
    "trendmicro.com",
    "google.com",
    "mozilla.org",
    "github.com",
    "gitlab.com",
)


def _first_existing_col(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    return None


def _request_with_retry(session, url, params=None, timeout=60, max_retries=4, base_delay=1.2):
    last_exc = None
    for attempt in range(max_retries):
        try:
            response = session.get(url, params=params, timeout=timeout)
            if response.status_code == 429:
                wait_s = min(30, base_delay * (2 ** attempt))
                time.sleep(wait_s)
                continue
            response.raise_for_status()
            return response
        except requests.RequestException as exc:
            last_exc = exc
            wait_s = min(30, base_delay * (2 ** attempt))
            time.sleep(wait_s)
    raise RuntimeError(f"Falha ao consultar {url}: {last_exc}")


def _pick_official_advisory_link(links):
    for link in links:
        domain = urlparse(link).netloc.lower().replace("www.", "")
        if any(domain == d or domain.endswith(f".{d}") for d in OFFICIAL_ADVISORY_DOMAINS):
            if "nvd.nist.gov" not in domain:
                return link
    return ""


def _clean_notes_text(text):
    cleaned = re.sub(r"https?://[^\s<>\"\[\]]+", "", text, flags=re.IGNORECASE)
    cleaned = re.sub(r"\(\s*\)", "", cleaned)
    cleaned = re.sub(r"\[\s*\]", "", cleaned)
    cleaned = re.sub(r"\(\s+(?=[A-Za-z])", "", cleaned)
    cleaned = re.sub(r"\s+", " ", cleaned)
    cleaned = re.sub(r"\s+([,.;:!?])", r"\1", cleaned)
    return cleaned.strip(" -:;,.()[]\t")


def parse_notes(notes):
    default = {
        "notes_link": "",
        "notes_has_patch": False,
        "notes_has_exploit": False,
        "notes_critical_infra": False,
        "notes_text": "",
    }

    if pd.isna(notes):
        return default

    text = str(notes).strip()
    if not text:
        return default

    links = [
        link.rstrip(".,);]\"")
        for link in re.findall(r"https?://[^\s<>\"\[\]]+", text, flags=re.IGNORECASE)
    ]
    notes_link = _pick_official_advisory_link(links)

    notes_text = _clean_notes_text(text)
    text_lower = notes_text.lower()

    patch_terms = ["patch", "mitigat", "workaround", "upgrade", "update", "fix", "remediation"]
    exploit_terms = [
        "exploit",
        "active",
        "in the wild",
        "poc",
        "weaponized",
        "ransomware",
        "zero-day",
        "zero day",
    ]
    critical_infra_terms = [
        "critical infrastructure",
        "industrial control",
        "ics",
        "scada",
        "operational technology",
        "ot",
    ]

    return {
        "notes_link": notes_link,
        "notes_has_patch": any(term in text_lower for term in patch_terms),
        "notes_has_exploit": any(term in text_lower for term in exploit_terms),
        "notes_critical_infra": any(term in text_lower for term in critical_infra_terms),
        "notes_text": notes_text,
    }


def download_kev_raw_df(session, kev_url, raw_output_path):
    response = _request_with_retry(session, kev_url, timeout=90)
    raw_output_path.write_bytes(response.content)
    return pd.read_csv(raw_output_path, encoding="utf-8-sig")


def normalize_kev_events(raw_df):
    df = raw_df.copy()
    df.columns = [str(c).strip() for c in df.columns]

    date_col = _first_existing_col(df, ["dateAdded", "Date Added", "date_added"])
    cve_col = _first_existing_col(df, ["cveID", "cveId", "CVE ID", "cve_id"])
    vendor_col = _first_existing_col(df, ["vendorProject", "vendor"])
    product_col = _first_existing_col(df, ["product", "Product"])
    due_col = _first_existing_col(df, ["dueDate", "due_date"])
    ransomware_col = _first_existing_col(df, ["knownRansomwareCampaignUse", "ransomware"])
    notes_col = _first_existing_col(df, ["notes", "Notes"])

    rename_map = {}
    if date_col:
        rename_map[date_col] = "date"
    if cve_col:
        rename_map[cve_col] = "cve_id"
    if vendor_col:
        rename_map[vendor_col] = "vendor"
    if product_col:
        rename_map[product_col] = "product"
    if due_col:
        rename_map[due_col] = "due_date"
    if ransomware_col:
        rename_map[ransomware_col] = "known_ransomware"
    if notes_col:
        rename_map[notes_col] = "notes"

    events_df = df.rename(columns=rename_map)

    required_cols = [
        "date",
        "cve_id",
        "vendor",
        "product",
        "due_date",
        "known_ransomware",
        "ransomware_flag",
        "days_to_due",
        "notes",
        "notes_link",
        "notes_has_patch",
        "notes_has_exploit",
        "notes_critical_infra",
        "notes_text",
    ]
    for col in ["date", "cve_id", "vendor", "product", "due_date", "known_ransomware", "notes"]:
        if col not in events_df.columns:
            events_df[col] = ""

    events_df["cve_id"] = events_df["cve_id"].astype(str).str.strip()
    events_df = events_df[events_df["cve_id"].str.startswith("CVE-")].copy()

    events_df["date"] = pd.to_datetime(events_df["date"], utc=True, errors="coerce")
    events_df["due_date"] = pd.to_datetime(events_df["due_date"], utc=True, errors="coerce")
    events_df = events_df[events_df["date"].notna()].copy()

    events_df["ransomware_flag"] = (
        events_df["known_ransomware"].fillna("").astype(str).str.strip().str.lower().isin(["known", "yes", "true"])
    ).astype(int)
    events_df["days_to_due"] = (events_df["due_date"] - events_df["date"]).dt.days.astype("Int64")

    events_df["date"] = events_df["date"].dt.strftime("%Y-%m-%d")
    events_df["due_date"] = events_df["due_date"].dt.strftime("%Y-%m-%d")

    for col in ["vendor", "product", "known_ransomware", "notes"]:
        events_df[col] = events_df[col].fillna("").astype(str).str.strip()

    parsed_notes_df = events_df["notes"].apply(parse_notes).apply(pd.Series)
    events_df = pd.concat([events_df, parsed_notes_df], axis=1)

    for col in ["notes_link", "notes_text"]:
        events_df[col] = events_df[col].fillna("").astype(str).str.strip()
    for col in ["notes_has_patch", "notes_has_exploit", "notes_critical_infra"]:
        events_df[col] = events_df[col].fillna(False).astype(bool)

    events_df = events_df[required_cols].sort_values(["date", "cve_id"]).reset_index(drop=True)
    return events_df


def build_daily_counts(events_df, continuous=True):
    sparse = (
        events_df.groupby("date", dropna=True)["cve_id"]
        .count()
        .rename("threat_count")
        .reset_index()
        .sort_values("date")
    )

    if not continuous or sparse.empty:
        return sparse

    all_days = pd.date_range(
        start=pd.to_datetime(sparse["date"]).min(),
        end=pd.to_datetime(sparse["date"]).max(),
        freq="D",
    )
    continuous_df = pd.DataFrame({"date": all_days.strftime("%Y-%m-%d")})
    continuous_df = continuous_df.merge(sparse, on="date", how="left")
    continuous_df["threat_count"] = continuous_df["threat_count"].fillna(0).astype(int)
    return continuous_df


def build_top_tables(events_df):
    by_vendor = (
        events_df[events_df["vendor"] != ""]
        .groupby("vendor")["cve_id"]
        .nunique()
        .rename("threat_count")
        .reset_index()
        .sort_values("threat_count", ascending=False)
    )

    by_product = (
        events_df[events_df["product"] != ""]
        .groupby("product")["cve_id"]
        .nunique()
        .rename("threat_count")
        .reset_index()
        .sort_values("threat_count", ascending=False)
    )

    return by_vendor, by_product


def _extract_nvd_metrics(cve_obj):
    severity = ""
    cvss_score = None
    metrics = cve_obj.get("metrics", {})
    for key in ["cvssMetricV31", "cvssMetricV30", "cvssMetricV2"]:
        vals = metrics.get(key, [])
        if not vals:
            continue
        metric = vals[0]
        severity = (
            metric.get("cvssData", {}).get("baseSeverity")
            or metric.get("baseSeverity")
            or ""
        )
        cvss_score = metric.get("cvssData", {}).get("baseScore")
        if severity or cvss_score is not None:
            break
    return severity, cvss_score


def _extract_nvd_description(cve_obj):
    for item in cve_obj.get("descriptions", []):
        if item.get("lang") == "en":
            return (item.get("value") or "").strip()
    return ""


def fetch_nvd_enrichment_for_cves(session, cve_ids, api_key="", delay_seconds=0.4, max_items=500):
    rows = []
    if api_key:
        session.headers.update({"apiKey": api_key})

    for cve_id in sorted(set(cve_ids))[:max_items]:
        params = {"cveId": cve_id}
        try:
            response = _request_with_retry(
                session,
                NVD_API_URL,
                params=params,
                timeout=45,
                max_retries=3,
            )
            data = response.json()
            vulns = data.get("vulnerabilities", [])
            if not vulns:
                continue

            cve_obj = vulns[0].get("cve", {})
            severity, cvss_score = _extract_nvd_metrics(cve_obj)
            rows.append(
                {
                    "cve_id": cve_id,
                    "nvd_published": cve_obj.get("published", ""),
                    "nvd_last_modified": cve_obj.get("lastModified", ""),
                    "nvd_severity": severity,
                    "nvd_cvss_score": cvss_score,
                    "nvd_description": _extract_nvd_description(cve_obj),
                }
            )
            time.sleep(delay_seconds)
        except Exception:
            continue

    return pd.DataFrame(rows)


def fetch_epss_for_cves(session, cve_ids, chunk_size=100):
    cves = sorted(set(cve_ids))
    rows = []

    for i in range(0, len(cves), chunk_size):
        chunk = cves[i : i + chunk_size]
        if not chunk:
            continue
        params = {"cve": ",".join(chunk)}
        try:
            response = _request_with_retry(
                session,
                EPSS_API_URL,
                params=params,
                timeout=45,
                max_retries=3,
            )
            data = response.json().get("data", [])
            for rec in data:
                cve_id = rec.get("cve", "")
                if not cve_id:
                    continue
                rows.append(
                    {
                        "cve_id": cve_id,
                        "epss": rec.get("epss", ""),
                        "epss_percentile": rec.get("percentile", ""),
                        "epss_date": rec.get("date", ""),
                    }
                )
        except Exception:
            continue

    return pd.DataFrame(rows)


def build_enriched_events(events_df, nvd_df=None, epss_df=None):
    enriched = events_df.copy()
    if nvd_df is not None and not nvd_df.empty:
        enriched = enriched.merge(nvd_df, on="cve_id", how="left")
    if epss_df is not None and not epss_df.empty:
        enriched = enriched.merge(epss_df, on="cve_id", how="left")
    return enriched


def run_optional_enrichment(enabled, label, fetch_fn, output_key, disabled_warning, empty_warning):
    if not enabled:
        return pd.DataFrame(), disabled_warning

    print(f"Enriquecimento {label} opcional ativado...")
    enrichment_df = fetch_fn()
    if enrichment_df.empty:
        return enrichment_df, empty_warning

    save_csv(enrichment_df, FILES[output_key])
    print(f"{FILES[output_key].name}: {len(enrichment_df):,} linhas")
    return enrichment_df, ""


def save_csv(df, path):
    df.to_csv(path, index=False, encoding="utf-8")

In [34]:
# 3) execução principal (centrada em KEV-only)
session = requests.Session()
session.headers.update({"User-Agent": "kev-daily-threats-pipeline/1.0"})

execution_warnings = []

print("Baixando KEV...")
kev_raw_df = download_kev_raw_df(session, KEV_CSV_URL, FILES["kev_raw"])
print(f"KEV bruto: {len(kev_raw_df):,} linhas")

threats_daily_events_df = normalize_kev_events(kev_raw_df)
threats_daily_events_df["urgent"] = (
    threats_daily_events_df["days_to_due"].notna()
    & (threats_daily_events_df["days_to_due"] <= 30)
).astype(bool)
save_csv(threats_daily_events_df, FILES["threats_daily_events"])
print(f"threats_daily_events.csv: {len(threats_daily_events_df):,} linhas")

threats_daily_counts_df = build_daily_counts(threats_daily_events_df, continuous=True)
save_csv(threats_daily_counts_df, FILES["threats_daily_counts"])
print(f"threats_daily_counts.csv: {len(threats_daily_counts_df):,} linhas")

threats_by_vendor_df, threats_by_product_df = build_top_tables(threats_daily_events_df)
save_csv(threats_by_vendor_df, FILES["threats_by_vendor"])
save_csv(threats_by_product_df, FILES["threats_by_product"])
print(f"threats_by_vendor.csv: {len(threats_by_vendor_df):,} linhas")
print(f"threats_by_product.csv: {len(threats_by_product_df):,} linhas")

run_nvd_enrichment = PIPELINE_MODE == "full" and RUN_NVD

enrich_nvd_df, nvd_warning = run_optional_enrichment(
    enabled=run_nvd_enrichment,
    label="NVD",
    fetch_fn=lambda: fetch_nvd_enrichment_for_cves(
        session=session,
        cve_ids=threats_daily_events_df["cve_id"].tolist(),
        api_key=NVD_API_KEY,
    ),
    output_key="enrich_nvd",
    disabled_warning="NVD desativado: fluxo principal executado em KEV-only.",
    empty_warning="NVD ativado, mas não retornou dados.",
)
if nvd_warning:
    execution_warnings.append(nvd_warning)

enrich_epss_df, epss_warning = run_optional_enrichment(
    enabled=RUN_EPSS,
    label="EPSS",
    fetch_fn=lambda: fetch_epss_for_cves(
        session=session,
        cve_ids=threats_daily_events_df["cve_id"].tolist(),
    ),
    output_key="enrich_epss",
    disabled_warning="EPSS desativado.",
    empty_warning="EPSS ativado, mas não retornou dados.",
)
if epss_warning:
    execution_warnings.append(epss_warning)

has_enrichment_data = any(not df.empty for df in [enrich_nvd_df, enrich_epss_df])
threats_daily_enriched_df = pd.DataFrame()

if has_enrichment_data:
    threats_daily_enriched_df = build_enriched_events(
        threats_daily_events_df,
        nvd_df=enrich_nvd_df,
        epss_df=enrich_epss_df,
    )
    save_csv(threats_daily_enriched_df, FILES["threats_daily_enriched"])
    print(f"threats_daily_enriched.csv: {len(threats_daily_enriched_df):,} linhas")

print("Execução principal finalizada.")

Baixando KEV...
KEV bruto: 1,554 linhas
threats_daily_events.csv: 1,554 linhas
threats_daily_counts.csv: 1,606 linhas
threats_by_vendor.csv: 256 linhas
threats_by_product.csv: 626 linhas
Execução principal finalizada.


## Análise e validação

Esta seção verifica se a base principal foi gerada corretamente no modo KEV-only e apresenta métricas úteis para operação, não apenas para inspeção exploratória.

### O que esta etapa confirma

- volume total de registros baixados do KEV
- volume total de eventos normalizados
- quantidade de dias únicos no histórico
- intervalo mínimo e máximo de datas disponíveis
- total de eventos marcados com `ransomware_flag`
- menor prazo observado em `days_to_due`
- principais vendors e products

A intenção aqui é validar qualidade operacional da base antes de partir para visualização ou enriquecimento adicional.

In [33]:
# 4) análise e validação
if "threats_daily_events_df" not in globals() or threats_daily_events_df.empty:
    raise RuntimeError("Execute a célula 4 (execução principal) antes desta análise.")

date_series = pd.to_datetime(threats_daily_events_df["date"], errors="coerce")
min_date = date_series.min()
max_date = date_series.max()

days_to_due_series = (
    threats_daily_events_df["days_to_due"].dropna()
    if "days_to_due" in threats_daily_events_df.columns
    else pd.Series(dtype="Int64")
)

metrics = {
    "rows_kev_raw": int(len(kev_raw_df)) if "kev_raw_df" in globals() else 0,
    "rows_threats_daily_events": int(len(threats_daily_events_df)),
    "rows_threats_daily_counts": int(len(threats_daily_counts_df)) if "threats_daily_counts_df" in globals() else 0,
    "unique_days": int(threats_daily_events_df["date"].nunique()),
    "ransomware_flag_sum": int(threats_daily_events_df["ransomware_flag"].sum()) if "ransomware_flag" in threats_daily_events_df.columns else 0,
    "urgent_sum": int(threats_daily_events_df["urgent"].sum()) if "urgent" in threats_daily_events_df.columns else 0,
    "min_days_to_due": None if days_to_due_series.empty else int(days_to_due_series.min()),
    "min_date": "" if pd.isna(min_date) else min_date.strftime("%Y-%m-%d"),
    "max_date": "" if pd.isna(max_date) else max_date.strftime("%Y-%m-%d"),
}

print("Métricas principais")
print(json.dumps(metrics, indent=2, ensure_ascii=False))

print("\nTop vendors")
display(threats_by_vendor_df.head(15))

print("\nTop products")
display(threats_by_product_df.head(15))

print("\nAmostra de eventos diários")
display(threats_daily_events_df.head(10))

print("\nAmostra de contagem diária")
display(threats_daily_counts_df.head(10))

Métricas principais
{
  "rows_kev_raw": 1554,
  "rows_threats_daily_events": 1554,
  "rows_threats_daily_counts": 1606,
  "unique_days": 397,
  "ransomware_flag_sum": 313,
  "urgent_sum": 1292,
  "min_days_to_due": 1,
  "min_date": "2021-11-03",
  "max_date": "2026-03-27"
}

Top vendors


,vendor,threat_count
113,Microsoft,362
13,Apple,93
32,Cisco,86
7,Adobe,76
75,Google,70
140,Oracle,42
12,Apache,38
89,Ivanti,32
225,VMware,26
42,D-Link,25



Top products


,product,threat_count
582,Windows,168
354,Multiple Products,79
99,Chromium V8,38
281,Internet Explorer,34
217,Flash Player,33
379,Office,28
308,Kernel,27
579,Win32k,25
199,Exchange Server,16
112,ColdFusion,15



Amostra de eventos diários


,date,cve_id,vendor,product,due_date,known_ransomware,ransomware_flag,days_to_due,notes,notes_link,notes_has_patch,notes_has_exploit,notes_critical_infra,notes_text,urgent
0,2021-11-03,CVE-2010-5326,SAP,NetWeaver,2022-05-03,Unknown,0,181,https://nvd.nist.gov/vuln/detail/CVE-2010-5326,,False,False,False,,False
1,2021-11-03,CVE-2012-0158,Microsoft,MSCOMCTL.OCX,2022-05-03,Unknown,0,181,https://nvd.nist.gov/vuln/detail/CVE-2012-0158,,False,False,False,,False
2,2021-11-03,CVE-2012-3152,Oracle,Fusion Middleware,2022-05-03,Unknown,0,181,https://nvd.nist.gov/vuln/detail/CVE-2012-3152,,False,False,False,,False
3,2021-11-03,CVE-2014-1812,Microsoft,Windows,2022-05-03,Known,1,181,https://nvd.nist.gov/vuln/detail/CVE-2014-1812,,False,False,False,,False
4,2021-11-03,CVE-2015-1641,Microsoft,Office,2022-05-03,Unknown,0,181,https://nvd.nist.gov/vuln/detail/CVE-2015-1641,,False,False,False,,False
5,2021-11-03,CVE-2015-4852,Oracle,WebLogic Server,2022-05-03,Unknown,0,181,https://nvd.nist.gov/vuln/detail/CVE-2015-4852,,False,False,False,,False
6,2021-11-03,CVE-2016-0167,Microsoft,Win32k,2022-05-03,Known,1,181,https://nvd.nist.gov/vuln/detail/CVE-2016-0167,,False,False,False,,False
7,2021-11-03,CVE-2016-0185,Microsoft,Windows,2022-05-03,Unknown,0,181,https://nvd.nist.gov/vuln/detail/CVE-2016-0185,,False,False,False,,False
8,2021-11-03,CVE-2016-3235,Microsoft,Office,2022-05-03,Unknown,0,181,https://nvd.nist.gov/vuln/detail/CVE-2016-3235,,False,False,False,,False
9,2021-11-03,CVE-2016-3643,SolarWinds,Virtualization Manager,2022-05-03,Unknown,0,181,https://nvd.nist.gov/vuln/detail/CVE-2016-3643,,False,False,False,,False



Amostra de contagem diária


,date,threat_count
0,2021-11-03,287
1,2021-11-04,0
2,2021-11-05,0
3,2021-11-06,0
4,2021-11-07,0
5,2021-11-08,0
6,2021-11-09,0
7,2021-11-10,0
8,2021-11-11,0
9,2021-11-12,0


In [16]:
# 5) visualizações simples (KEV-only)
if "threats_daily_counts_df" not in globals() or threats_daily_counts_df.empty:
    raise RuntimeError("Execute a célula 4 (execução principal) antes dos gráficos.")

# Série temporal diária de ameaças
fig_daily = px.line(
    threats_daily_counts_df,
    x="date",
    y="threat_count",
    title="Ameaças adicionadas por dia (KEV)",
)
fig_daily.write_html(PLOTS_DIR / "01_threats_daily_timeline.html")

plt.figure(figsize=(12, 4))
plt.plot(pd.to_datetime(threats_daily_counts_df["date"]), threats_daily_counts_df["threat_count"], linewidth=1.2)
plt.title("Ameaças adicionadas por dia (KEV)")
plt.xlabel("Data")
plt.ylabel("Quantidade")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "01_threats_daily_timeline.png", dpi=140)
plt.close()

# Top vendors
top_vendor_plot = threats_by_vendor_df.head(20)
if not top_vendor_plot.empty:
    fig_vendor = px.bar(
        top_vendor_plot,
        x="vendor",
        y="threat_count",
        title="Top 20 vendors por ameaças",
    )
    fig_vendor.update_layout(xaxis_tickangle=-45)
    fig_vendor.write_html(PLOTS_DIR / "02_threats_top_vendors.html")

# Top products
top_product_plot = threats_by_product_df.head(20)
if not top_product_plot.empty:
    fig_product = px.bar(
        top_product_plot,
        x="product",
        y="threat_count",
        title="Top 20 products por ameaças",
    )
    fig_product.update_layout(xaxis_tickangle=-45)
    fig_product.write_html(PLOTS_DIR / "03_threats_top_products.html")

print(f"Gráficos salvos em: {PLOTS_DIR.resolve()}")

Gráficos salvos em: /Users/gabe/graphos/plots


In [17]:
# 6) resumo final
summary = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "pipeline_mode": PIPELINE_MODE,
    "run_nvd": RUN_NVD,
    "run_epss": RUN_EPSS,
    "rows": {
        "kev_raw": int(len(kev_raw_df)) if "kev_raw_df" in globals() else 0,
        "threats_daily_events": int(len(threats_daily_events_df)) if "threats_daily_events_df" in globals() else 0,
        "threats_daily_counts": int(len(threats_daily_counts_df)) if "threats_daily_counts_df" in globals() else 0,
        "threats_by_vendor": int(len(threats_by_vendor_df)) if "threats_by_vendor_df" in globals() else 0,
        "threats_by_product": int(len(threats_by_product_df)) if "threats_by_product_df" in globals() else 0,
        "enrich_nvd": int(len(enrich_nvd_df)) if "enrich_nvd_df" in globals() else 0,
        "enrich_epss": int(len(enrich_epss_df)) if "enrich_epss_df" in globals() else 0,
        "threats_daily_enriched": int(len(threats_daily_enriched_df)) if "threats_daily_enriched_df" in globals() else 0,
    },
    "date_range": {
        "min_date": "" if "threats_daily_events_df" not in globals() or threats_daily_events_df.empty else str(threats_daily_events_df["date"].min()),
        "max_date": "" if "threats_daily_events_df" not in globals() or threats_daily_events_df.empty else str(threats_daily_events_df["date"].max()),
    },
    "warnings": execution_warnings if "execution_warnings" in globals() else [],
    "files": {
        name: str(path) for name, path in FILES.items() if Path(path).exists()
    },
}

FILES["summary"].write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")

print("Resumo salvo em summary.json")
print("Arquivos gerados:")
for name, path in summary["files"].items():
    print(f"- {name}: {path}")

Resumo salvo em summary.json
Arquivos gerados:
- kev_raw: kev_raw.csv
- threats_daily_events: threats_daily_events.csv
- threats_daily_counts: threats_daily_counts.csv
- threats_by_vendor: threats_by_vendor.csv
- threats_by_product: threats_by_product.csv
- summary: summary.json


## Enriquecimento opcional com NVD e EPSS

Esta seção existe para aprofundamento analítico, não para sustentar a execução principal.

## Decisão de projeto

- O pipeline diário continua válido mesmo sem NVD e EPSS.
- O enriquecimento foi mantido fora do caminho crítico para reduzir acoplamento com APIs externas.
- Falhas de enriquecimento geram aviso, mas não invalidam os CSVs principais.

## Quando usar

- Use NVD quando precisar de contexto técnico adicional como severidade, CVSS e descrição.
- Use EPSS quando precisar de uma visão de probabilidade relativa de exploração.
- Ignore ambos quando o objetivo for apenas operar a fila diária de ameaças com base em KEV, prazo e ransomware.

In [18]:
# 7) validação da seção opcional de enriquecimento
if PIPELINE_MODE != "full" and not RUN_EPSS:
    print("Enriquecimentos opcionais desativados (comportamento esperado no modo KEV-only).")
else:
    if "enrich_nvd_df" in globals() and not enrich_nvd_df.empty:
        print(f"NVD enrichment disponível: {len(enrich_nvd_df):,} linhas")
        display(enrich_nvd_df.head(10))
    else:
        print("NVD sem dados nesta execução (ou desativado).")

    if "enrich_epss_df" in globals() and not enrich_epss_df.empty:
        print(f"EPSS enrichment disponível: {len(enrich_epss_df):,} linhas")
        display(enrich_epss_df.head(10))
    else:
        print("EPSS sem dados nesta execução (ou desativado).")

    if "threats_daily_enriched_df" in globals() and not threats_daily_enriched_df.empty:
        print(f"Base enriquecida disponível: {len(threats_daily_enriched_df):,} linhas")
        display(threats_daily_enriched_df.head(10))

Enriquecimentos opcionais desativados (comportamento esperado no modo KEV-only).
